In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.runnables import RunnableConfig
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv

In [4]:
load_dotenv()
runnableConfig = RunnableConfig(retries=3)
llm_model = ChatOpenRouter(model="meta-llama/llama-3-8b-instruct")

In [5]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str

In [6]:
def generate_outline(state: BlogState) -> BlogState:
    topic = state['topic']

    prompt = f'''
You're a professional blog writer. Give a blog outline in around 5 words for the below topic.
Topic: {topic}
'''
    
    outline = llm_model.invoke(prompt).content

    state['outline'] = outline
    
    return state

In [7]:
def generate_blog(state: BlogState) -> BlogState:
    topic = state['topic']

    prompt = f'''
You're an experienced blog writer. You need to write a blog in around 100 words for the below topic.
Topic: {topic}
'''
    
    content = llm_model.invoke(prompt).content

    state['content'] = content
    
    return state

In [8]:
graph = StateGraph(BlogState)

graph.add_node('generate_outline', generate_outline)
graph.add_node('generate_blog', generate_blog)

graph.add_edge(START, 'generate_blog')
graph.add_edge('generate_outline', 'generate_blog')
graph.add_edge('generate_blog', END)

workflow = graph.compile()

In [9]:
input_state = {'topic': 'Artificial Intelligence'}
output_state = workflow.invoke(input_state)

print(output_state['content'])

Here is a 100-word blog on the topic of Artificial Intelligence:

**The Rise of Artificial Intelligence: Revolutionizing Our World**

Artificial Intelligence (AI) is no longer the stuff of science fiction. Its rapid advancement is transforming industries, improving efficiency, and enhancing our daily lives. AI-powered virtual assistants like Siri, Alexa, and Google Assistant are now an integral part of our daily routines. But AI's impact goes beyond voice assistants. Automating tasks, predicting traffic flow, and analyzing medical images are just a few examples of AI's vast capabilities. As AI technology continues to evolve, we can expect to see increased innovation in fields like healthcare, finance, and education. The future is intelligent, and AI is leading the charge.
